# 🎧 MP3 Compression Simulation (Educational Version)

This notebook demonstrates a simplified simulation of how MP3 compresses audio.

⚠️ This is NOT a real MP3 encoder.
It demonstrates the core ideas:

1. Block-based transform coding
2. Frequency-domain masking (psychoacoustic concept)
3. Quantization (lossy step)
4. Reconstruction
5. Visual comparison


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
from scipy.fft import fft, ifft

print('Libraries loaded successfully.')

## Load Audio File (WAV recommended)

In [ ]:
file_path = 'original.wav'  # Change to your file

audio, sr = librosa.load(file_path, sr=None, mono=True)

print('Sample rate:', sr)
print('Duration:', len(audio)/sr, 'seconds')

## Step 1: Block-Based Transform (FFT approximation of MDCT)

In [ ]:
frame_size = 1024
hop_size = 512

frames = librosa.util.frame(audio, frame_length=frame_size, hop_length=hop_size).T
window = np.hanning(frame_size)

transformed = []

for frame in frames:
    windowed = frame * window
    spectrum = fft(windowed)
    transformed.append(spectrum)

transformed = np.array(transformed)

print('Transformation complete.')

## Step 2: Simulated Psychoacoustic Masking

In [ ]:
masked = transformed.copy()

masking_ratio = 0.05  # Increase for stronger compression

for i in range(len(masked)):
    magnitude = np.abs(masked[i])
    threshold = masking_ratio * np.max(magnitude)
    masked[i][magnitude < threshold] = 0

print('Masking applied.')

## Step 3: Quantization (Lossy Step)

In [ ]:
quantization_step = 0.02  # Increase for lower quality

quantized = np.round(masked.real / quantization_step) * quantization_step + \
            1j * np.round(masked.imag / quantization_step) * quantization_step

print('Quantization complete.')

## Step 4: Reconstruct Audio

In [ ]:
reconstructed = np.zeros(len(audio))

for i in range(len(quantized)):
    frame_time = np.real(ifft(quantized[i]))
    start = i * hop_size
    reconstructed[start:start+frame_size] += frame_time * window

reconstructed = reconstructed / np.max(np.abs(reconstructed))

print('Reconstruction complete.')

## Waveform Comparison

In [ ]:
plt.figure()
plt.plot(audio[:2000])
plt.title('Original Waveform (First 2000 samples)')
plt.xlabel('Sample Index')
plt.ylabel('Amplitude')
plt.show()

plt.figure()
plt.plot(reconstructed[:2000])
plt.title('Compressed Waveform (First 2000 samples)')
plt.xlabel('Sample Index')
plt.ylabel('Amplitude')
plt.show()

## Spectrogram Comparison

In [ ]:
plt.figure()
librosa.display.specshow(
    librosa.amplitude_to_db(np.abs(librosa.stft(audio)), ref=np.max),
    sr=sr,
    x_axis='time',
    y_axis='hz'
)
plt.title('Original Spectrogram')
plt.colorbar()
plt.show()

plt.figure()
librosa.display.specshow(
    librosa.amplitude_to_db(np.abs(librosa.stft(reconstructed)), ref=np.max),
    sr=sr,
    x_axis='time',
    y_axis='hz'
)
plt.title('Compressed Spectrogram')
plt.colorbar()
plt.show()